# 01-reach: a two-wheeler OSRM graph, candidate sites and a travel-time table

> **Stage 0.5 record, kept as run.** The pipeline code now lives in the package (`src/catchment/`), and the
> OSRM script and two-wheeler profile have moved from `notebooks/` to `osrm/` (`osrm/osrm.sh`, `osrm/two-wheeler.lua`).

How long a rider takes between two points, and so which hexes a store could reach. OSRM (the open-source routing
engine) runs in Docker on the clipped OSM extract with a two-wheeler profile, and answers **free-flow** routes: no
traffic, no queues. **Travel time = free-flow time × congestion multiplier** (a lever, 1.0 to 2.0); here only the
free-flow part exists.

**Part 1** builds the graph on the 8 GB machine, records peak memory and time per build step, and queries a handful of
routes on the slice as a sanity check (not the stage 1 spot-check). **Part 2** picks the candidate sites, builds the
travel-time table of candidate site × hex pairs within 17 min free-flow on the slice, checks how well points snap to
the road, and extrapolates the table to the full area at H3 res 9 and res 8.

In [ ]:
import json
import re
import subprocess
import os
import urllib.request

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyproj import Geod, Transformer
from shapely.geometry import LineString, Point, box

from catchment.area import load_area
from catchment.fetch import DATA

ROOT = DATA.parent
NB = ROOT / "notebooks"
OSRM_DIR = DATA.resolve() / "osrm"
OSRM = "http://localhost:5001"
IMAGE = "ghcr.io/project-osrm/osrm-backend:v26.10.0-debian"   # same pin as notebooks/osrm.sh
FIGS = NB / "figures"
FIGS.mkdir(parents=True, exist_ok=True)
area = load_area()

## Assumptions

Every edit to the profile `notebooks/two-wheeler.lua`, a copy of OSRM's `car.lua` from the pinned image, in one
place; each is commented in the file with `-- two-wheeler:`. OSRM ships no two-wheeler profile and no maintained
community one exists. **Verified** means checked against the source on the date shown; **Assumption** means a
labelled choice with no external source. Anything not listed is as in `car.lua`.

In [ ]:
def lua_speeds(text):
    """highway class -> km/h from the `speeds = Sequence { highway = {...} }` block of an OSRM profile."""
    block = text[text.index("speeds = Sequence"):text.index("service_penalties")]
    return {k: int(v) for k, v in re.findall(r"^\s*(\w+)\s*=\s*(\d+)\s*,?\s*(?:--.*)?$", block, flags=re.M)}

car_lua = subprocess.run(["docker", "run", "--rm", IMAGE, "cat", "/opt/car.lua"], check=True, capture_output=True, text=True).stdout
two_lua = (NB / "two-wheeler.lua").read_text()
car_speeds, two_speeds = lua_speeds(car_lua), lua_speeds(two_lua)
speed_table = pd.DataFrame({"car.lua (km/h)": car_speeds, "two-wheeler.lua (km/h)": two_speeds})
speed_table.index.name = "highway class"
assert list(speed_table.index) == list(car_speeds) and speed_table.notna().all().all()

docker_mem = int(subprocess.run(["docker", "info", "--format", "{{.MemTotal}}"], check=True, capture_output=True, text=True).stdout)
host_ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
GIB = 2**30

In [ ]:
# --- Assumptions ---------------------------------------------------------
assumptions = pd.DataFrame(
    [
        ("Road-class speeds", "Lower than car.lua in every class: " + ", ".join(f"{k} {speed_table.loc[k, 'car.lua (km/h)']} to {speed_table.loc[k, 'two-wheeler.lua (km/h)']}" for k in ["motorway", "primary", "residential", "service"]) + " km/h (table below)", "No external source; a delivery rider's free-flow speed, guessed. Gurugram's real speeds are mostly set by traffic, which the congestion multiplier stands in for (decision log, 2026-10-29)", "-", "Assumption"),
        ("Motorcycle access", "`motorcycle` is the most specific access tag (then motor_vehicle, vehicle, access) and replaces `motorcar`, so motorcycle=no / yes decide access; also read for oneway:motorcycle and for restriction and except=motorcycle on turn restrictions", "OSM access-tag hierarchy, https://wiki.openstreetmap.org/wiki/Key:access. The extract has about 250 ways with a motorcycle tag, mostly motorcycle=no (counted with osmium on the clipped extract). Whether the expressway carriageways ban two-wheelers is not checked; where unmapped, a two-wheeler may use any road a car can", "2026-10-29", "Unverified"),
        ("maxspeed tags", "Can only lower the class speed, never raise it (still × speed_reduction 0.8, as in car.lua)", "In car.lua a maxspeed tag overwrites the class speed, so a tagged 80 km/h on a residential road would beat the two-wheeler class speed; the profile wraps the handler to cap it. The extract has maxspeed values up to 120. Design choice, not a fact", "-", "Assumption"),
        ("Dimensions, penalties, turns", "Unchanged: vehicle size limits, narrow-road and service-road penalties, turn penalties, barriers, u-turn penalty 20 s", "car.lua kept as is; a rider fits where a car does and probably filters through more, so some times may be high", "-", "Assumption"),
        ("Algorithm", "Contraction hierarchies (osrm-extract, osrm-contract, osrm-routed --algorithm ch)", "Fastest many-to-many tables, which the travel-time table needs; MLD would need osrm-partition and osrm-customize", "-", "Assumption"),
        ("OSRM image", IMAGE, "https://github.com/Project-OSRM/osrm-backend/pkgs/container/osrm-backend, linux/arm64 build (plain v26.10.0 tag does not exist)", "2026-10-29", "Verified"),
        ("Docker Desktop memory limit", f"{docker_mem / GIB:.2f} GiB ({docker_mem:,} bytes) of {host_ram / GIB:.0f} GiB host RAM", "`docker info --format '{{.MemTotal}}'` and `os.sysconf`, measured when this notebook runs", "2026-10-29", "Verified"),
    ],
    columns=["Input", "Value", "Source", "Date", "Status"],
)
pd.set_option("display.max_colwidth", None)
display(assumptions.style.hide(axis="index").set_properties(**{"text-align": "left"}))
display(speed_table)

The speeds are read from the two Lua files, not copied by hand. They are free-flow, so a car's 90 km/h on a
motorway becomes 60 for a rider: roughly a loaded delivery bike on an open road. On a service road or in a
living street the rider crawls. These values are guesses to be revisited when the stage 1 spot-check shows how far
the free-flow times are from a map app's.

## Build

`notebooks/osrm.sh build` runs `osrm-extract` then `osrm-contract` on `data/osm.osm.pbf` (the extract, clipped to the
area plus 5 km), each in its own container. The script records each step's peak memory (the container's cgroup
`memory.peak`, which counts page cache too, so it is an upper bound on the process's own memory) and wall time in
`data/osrm/build-stats.tsv`, and the step logs next to it. The graph stays in
`data/osrm/` and is not committed. It rebuilds each time, about two minutes.

In [ ]:
build = subprocess.run([str(NB / "osrm.sh"), "build"], capture_output=True, text=True)
if build.returncode:
    print(build.stderr)
    build.check_returncode()

stats = pd.read_csv(OSRM_DIR / "build-stats.tsv", sep="\t")
stats["peak MB"] = (stats.peak_bytes / 1e6).round(0)
stats["seconds"] = stats.seconds.round(1)
display(stats[["step", "peak MB", "seconds"]].style.hide(axis="index"))

pbf_mb = (DATA / "osm.osm.pbf").stat().st_size / 1e6
extract_log = (OSRM_DIR / "extract.log").read_text()
contract_log = (OSRM_DIR / "contract.log").read_text()
raw = re.search(r"Raw input contains (\d+) nodes, (\d+) ways, and (\d+) relations, (\d+) restrictions", extract_log).groups()
edges = int(re.search(r"Contracted graph has (\d+) edges", contract_log).group(1))
problems = [l for l in (extract_log + contract_log).splitlines() if re.search(r"\[(warning|error)\]", l, re.I)]
print(f"Input pbf: {pbf_mb:.1f} MB; {int(raw[0]):,} nodes, {int(raw[1]):,} ways, {int(raw[2]):,} relations, {int(raw[3])} turn restrictions")
print(f"Contracted graph: {edges:,} edges; graph files in data/osrm/: {sum(f.stat().st_size for f in OSRM_DIR.glob('osm.osrm*')) / 1e6:.0f} MB")
print(f"Warnings or errors in the step logs: {len(problems)}")
for l in problems[:10]:
    print(" ", l)
print(f"Peak memory vs the Docker limit: {stats.peak_bytes.max() / docker_mem:.0%}")

## Serve and query routes

`osrm-routed` runs as the container `catchment-osrm` on `localhost:5001` (macOS often holds 5000). The slice is the
same one `00-demand` uses: a lon/lat box cut to the area. Seven routes inside it, short to long, between old
Gurugram (sectors 4-14, Sadar) and the Golf Course Road / Sohna Road side. The endpoints are coordinates of Overture
places with the named area in their name; "near" means the coordinates are one such place's, not a surveyed landmark.

In [ ]:
slice_poly = box(77.005, 28.405, 77.105, 28.480).intersection(area)

# short name -> (lon, lat); every point is the position of an Overture place named for the area
POINTS = {
    "Sadar Bazar": (77.0293, 28.4612),
    "Civil Lines": (77.0476, 28.4630),
    "Rajiv Chowk": (77.0322, 28.4501),
    "Medanta": (77.0569, 28.4712),
    "Sikanderpur": (77.1026, 28.4752),
    "Sector 56": (77.0983, 28.4250),
    "Sohna Rd, sec 48": (77.0400, 28.4114),
    "Golf Course Rd, sec 57": (77.0740, 28.4151),
}
for name, (x, y) in POINTS.items():
    assert slice_poly.contains(Point(x, y)), f"{name} is outside the slice"

ROUTES = [  # (from, to)
    ("Sadar Bazar", "Civil Lines"),
    ("Rajiv Chowk", "Sadar Bazar"),
    ("Sadar Bazar", "Medanta"),
    ("Sohna Rd, sec 48", "Golf Course Rd, sec 57"),
    ("Medanta", "Sikanderpur"),
    ("Sadar Bazar", "Sector 56"),
    ("Sohna Rd, sec 48", "Sikanderpur"),
]

In [ ]:
geod = Geod(ellps="WGS84")
to_utm = Transformer.from_crs(4326, 32643, always_xy=True)   # metres, for distances along the route

def route(a, b):
    (x1, y1), (x2, y2) = POINTS[a], POINTS[b]
    url = f"{OSRM}/route/v1/driving/{x1},{y1};{x2},{y2}?overview=full&geometries=geojson&annotations=speed"
    with urllib.request.urlopen(url, timeout=30) as r:
        res = json.load(r)
    assert res["code"] == "Ok", res
    rt = res["routes"][0]
    _, _, straight_m = geod.inv(x1, y1, x2, y2)
    # path length ridden while still within 100 m of the start, and after first coming within 100 m of the end;
    # about 100 m when the route leaves and arrives cleanly, much more if it runs out and back
    x, y = to_utm.transform(*np.array(rt["geometry"]["coordinates"]).T)
    along = np.r_[0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]
    ax, ay = to_utm.transform(x1, y1)
    bx, by = to_utm.transform(x2, y2)
    leave = along[np.where(np.hypot(x - ax, y - ay) <= 100)[0].max()]
    arrive = along[-1] - along[np.where(np.hypot(x - bx, y - by) <= 100)[0].min()]
    return {
        "from": a, "to": b,
        "free-flow min": rt["duration"] / 60,
        "road km": rt["distance"] / 1000,
        "straight km": straight_m / 1000,
        "detour": rt["distance"] / straight_m,
        "avg km/h": rt["distance"] / rt["duration"] * 3.6,
        "top km/h": max(rt["legs"][0]["annotation"]["speed"]) * 3.6,  # annotation speed is m/s
        "snap m": max(w["distance"] for w in res["waypoints"]),
        "start/end path m": max(leave, arrive),
        "geometry": LineString(rt["geometry"]["coordinates"]),
    }

_ = subprocess.run([str(NB / "osrm.sh"), "serve"], check=True)
try:
    routes = pd.DataFrame([route(a, b) for a, b in ROUTES])
finally:   # stop the server even if a query fails
    _ = subprocess.run([str(NB / "osrm.sh"), "stop"], check=True)
routes.index = range(1, len(routes) + 1)
routes.index.name = "route"
show = routes.drop(columns="geometry").round({"free-flow min": 1, "road km": 1, "straight km": 1, "detour": 2, "avg km/h": 1, "top km/h": 0, "snap m": 0, "start/end path m": 0})
display(show)

`top km/h` is the fastest annotated segment on the route (OSRM reports segment speed in m/s; converted); with the
two-wheeler profile it can never exceed the motorway class's 60. `snap m` is how far the farthest endpoint sat from the nearest road.

`start/end path m` is the larger of two path lengths: the distance ridden before the route finally leaves the 100 m
around the start, and the distance ridden after it first reaches the 100 m around the destination. About 100 m or less
means a clean departure and arrival. A snap of under 50 m to the nearest road can still land on the far carriageway
of a divided road or on a one-way, and then the route has to ride to the next turn or U-turn and double
back. That inflates short routes' times and detour ratios.

In [ ]:
bad_speed = routes[(routes["avg km/h"] < 15) | (routes["avg km/h"] > 35)]
bad_detour = routes[(routes.detour < 1.2) | (routes.detour > 1.8)]
bad_snap = routes[routes["snap m"] > 100]
bad_loop = routes[routes["start/end path m"] > 300]
print("Plausible: average 15-35 km/h, detour ratio 1.2-1.8, endpoints within 100 m of a road, no out-and-back at an endpoint.")
print("Average speed outside 15-35 km/h:", list(bad_speed.index) or "none")
print("Detour ratio outside 1.2-1.8:", list(bad_detour.index) or "none")
print("Endpoint more than 100 m from a road:", list(bad_snap.index) or "none")
print("Route runs out and back at an endpoint (more than 300 m within or after 100 m of it):", list(bad_loop.index) or "none")
print(f"Top segment speed across routes: {routes['top km/h'].max():.0f} km/h (class cap 60)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)
gpd.GeoSeries([slice_poly], crs=4326).plot(ax=ax, facecolor="#f2f2f2", edgecolor="black", linewidth=1)
colors = plt.cm.tab10(np.arange(len(routes)))
for (i, r), c in zip(routes.iterrows(), colors):
    x, y = r.geometry.xy
    ax.plot(x, y, color=c, linewidth=2.5, alpha=0.9,
            label=f"{i}  {r['from']} to {r['to']}: {show.loc[i, 'free-flow min']:.1f} min, {show.loc[i, 'road km']:.1f} km")  # same rounding as the table
    mid = r.geometry.interpolate(0.5, normalized=True)
    ax.annotate(f"{r['free-flow min']:.0f}", (mid.x, mid.y), ha="center", va="center", fontsize=9, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=c, lw=1.5))
for name, (x, y) in POINTS.items():
    ax.plot(x, y, "o", color="black", markersize=6, zorder=5)
    ax.annotate(name, (x, y), xytext=(5, 5), textcoords="offset points", fontsize=8, zorder=6)
ax.set_aspect(1 / np.cos(np.radians(28.44)))
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8, ncol=1, frameon=False)
ax.set_title("Free-flow two-wheeler routes on the slice (labels are minutes)")
fig.savefig(FIGS / "01-routes.png", dpi=120, bbox_inches="tight")
plt.show()

![Free-flow two-wheeler routes on the slice](figures/01-routes.png)

Results from the run on 2026-10-29 (copied here because outputs are stripped on commit; re-running updates the
printed tables, not this one):

Docker Desktop limit 3.82 GiB (4,106,604,544 bytes) on an 8 GiB host, image `ghcr.io/project-osrm/osrm-backend:v26.10.0-debian`
(arm64), CH algorithm. Input `data/osm.osm.pbf` 16.6 MB: 1,913,652 nodes, 358,163 ways, 694 relations, 102 turn
restrictions. No warnings or errors in the step logs.

| step | peak memory (MB) | wall time (s) |
|---|---:|---:|
| `osrm-extract` | 491 | 18.3 |
| `osrm-contract` | 214 | 187.7 |

Peak memory is the container's cgroup `memory.peak`, which counts page cache too, so it is an upper bound on the
process's own memory. It is 12% of Docker's limit, so the clipped extract builds with plenty of room. The graph (1.81 M
contracted edges) takes 117 MB on disk. Other builds the same day took 16-24 s and 101-188 s: wall time moves with
machine load, peak memory barely does. Whether the full area fits is a stage 1 question; this extract is already the
area plus a 5 km buffer.

| route | from | to | free-flow min | road km | straight km | detour | avg km/h | start/end path m |
|---:|---|---|---:|---:|---:|---:|---:|---:|
| 1 | Sadar Bazar | Civil Lines | 7.2 | 2.9 | 1.8 | 1.59 | 24.0 | 105 |
| 2 | Rajiv Chowk | Sadar Bazar | 5.4 | 2.5 | 1.3 | 2.02 | 28.6 | 1,067 |
| 3 | Sadar Bazar | Medanta | 7.7 | 3.8 | 2.9 | 1.30 | 29.8 | 91 |
| 4 | Sohna Rd, sec 48 | Golf Course Rd, sec 57 | 10.0 | 5.3 | 3.4 | 1.57 | 31.5 | 97 |
| 5 | Medanta | Sikanderpur | 13.1 | 6.9 | 4.5 | 1.54 | 31.6 | 82 |
| 6 | Sadar Bazar | Sector 56 | 20.3 | 9.2 | 7.9 | 1.17 | 27.3 | 90 |
| 7 | Sohna Rd, sec 48 | Sikanderpur | 23.8 | 12.5 | 9.4 | 1.34 | 31.5 | 52 |

Sanity check: every average speed is inside 15-35 km/h, and every endpoint snapped to within 49 m of a road. Two
detour ratios fall outside 1.2-1.8: route 2 (2.02) and route 6 (1.17). Route 6 is nearly straight, which is allowed.
Route 2 is a snapping artefact: its start sits 49 m from the nearest road, but the route first rides about 480 m
south, away from the destination, turns around and comes back, so 1,067 m of its 2,548 m are spent at the start.
The other routes leave and arrive within about 100 m. This looks like the start landing on the far carriageway of a
divided road (Sohna Road) or on a one-way, so the route runs to the next U-turn; it is not checked against the OSM
tags. It inflates short routes' times and detour ratios (route 2 would probably be shorter than 5.4 min from the right
carriageway). **Open point for #4:** hex centroids and candidate sites snapped onto dual carriageways will do the same
thing, and short pairs are the ones a 17-min table cares about most; snapping is not fixed here. OSRM's per-segment
speeds cluster at the two-wheeler class values (20, 30, 35, 40, 50 km/h), not car.lua's (25, 40, 55, 65, 85), so the
profile edits took effect. Free-flow times are model output, not checked against a map app yet (stage 1 spot-check);
they include turn penalties, so the averages sit below the class speeds.

## Part 2: candidate sites and the travel-time table

Where a store could open, and which hexes it could reach. A **candidate site** is the centre of one of the area's
busiest hexes by count of mapped places, snapped to the road network. The **travel-time table** holds free-flow time
for every candidate site × hex pair within 17 minutes, which is every pair any scenario can use. This part decides how
many candidate sites to use (X), measures the table on the slice and extrapolates it to the full area, at
H3 res 9 and res 8, so the hex resolution can be chosen on numbers. It reads the hexes `00-demand` wrote, and starts
and stops its own OSRM server.

In [ ]:
import math
import time

import h3
from matplotlib.colors import PowerNorm
from shapely.geometry import Polygon

EXPLORE = DATA / "explore"
RES = (9, 8)

# --- Assumptions ---------------------------------------------------------
PICK_AND_PACK = 3                       # minutes, fixed
PROMISE, MULTIPLIER = 10, 1.4           # default scenario
MAX_PROMISE = 20                        # longest promise lever
MAX_PAIR_MIN = MAX_PROMISE - PICK_AND_PACK   # free-flow minutes kept in the table: 17
X_GRID = [2.5, 5, 10, 15, 20, 30]       # % of the area's hexes that become candidate sites
REACH_TARGET = 0.99                     # share of the slice's demand the slice's candidates must reach at the default

def reachable(ff_min):
    """Hex reachable from a site: free-flow minutes x multiplier + pick-and-pack <= promise (default scenario)."""
    return ff_min * MULTIPLIER + PICK_AND_PACK <= PROMISE + 1e-9

DEFAULT_PAIR_MIN = (PROMISE - PICK_AND_PACK) / MULTIPLIER   # 5.0 free-flow minutes

### Assumptions

Same convention as part 1. The two-wheeler speeds and every other part 1 assumption still apply.

In [ ]:
overture = pd.Series(json.load(open(DATA / "manifest.json"))["overture"])
assumptions2 = pd.DataFrame(
    [
        ("Candidate site rule", "One per hex, for the top X% of the area's hexes by Overture places count; the hex centre snapped to the nearest road (OSRM `/nearest`). Hexes with no places are never candidates; ties are broken by demand", "CONTEXT.md and decision log, 2026-10-29 (Explore); the tie-break and the no-places rule are this notebook's", "-", "Assumption"),
        ("Places count", "Overture places in the hex, except those marked permanently_closed; `operating_status` is empty for nearly all", f"Overture Maps places, release {overture.release}, `data/overture_places.parquet` (not stored in the repo)", overture.fetched, "Verified"),
        ("Pick-and-pack allowance", f"{PICK_AND_PACK} min, fixed", "Decision log, 2026-10-29 (Explore)", "2026-10-29", "Assumption"),
        ("Default scenario", f"Promise {PROMISE} min, congestion multiplier {MULTIPLIER}: a hex is reachable if free-flow min × {MULTIPLIER} + {PICK_AND_PACK} ≤ {PROMISE}, that is free-flow ≤ {DEFAULT_PAIR_MIN:.1f} min", "Decision log, 2026-10-29 (Explore)", "2026-10-29", "Assumption"),
        ("Pairs kept in the table", f"Free-flow ≤ {MAX_PAIR_MIN} min (longest promise {MAX_PROMISE} − {PICK_AND_PACK} at multiplier 1.0)", "Decision log, 2026-10-29 (Explore)", "2026-10-29", "Assumption"),
        ("X grid and rule", f"X in {X_GRID} % of area hexes. X is the smallest grid value whose slice candidates reach at least {REACH_TARGET:.0%} of the slice's demand at the default scenario; else the largest, with a warning", "No external source; chosen so that the slice is covered without candidates the table budget cannot afford", "-", "Assumption"),
        ("Travel-time source", "OSRM `/table` free-flow durations: sources are the snapped candidate sites, destinations the hex centres (OSRM snaps them to the nearest road).", "OSRM HTTP API, https://project-osrm.org/docs/v5.24.0/api/#table-service", "2026-10-29", "Verified"),
    ],
    columns=["Input", "Value", "Source", "Date", "Status"],
)
display(assumptions2.style.hide(axis="index").set_properties(**{"text-align": "left"}))

### Candidate sites

Places are counted per hex by point (`h3.latlng_to_cell`), at res 9 and at res 8 separately, so a res-8 hex's rank is by
its own count, not its children's. Hexes are ranked by places count, ties by demand; the top X% of the area's hexes
are candidate sites. Candidates for a smaller X are always a subset of those for a larger one, so the largest X in the
grid is ranked and snapped once. A hex and its candidate site lie inside the area; only roads extend into the buffer.

In [ ]:
hexes = {res: pd.read_parquet(EXPLORE / f"hexes-r{res}.parquet") for res in RES}
places = gpd.read_parquet(DATA / "overture_places.parquet")
print(f"Overture places: {len(places):,}; permanently_closed: {int((places.operating_status == 'permanently_closed').sum())}")
places = places[places.operating_status != "permanently_closed"]

for res, hx in hexes.items():
    cells = pd.Series([h3.latlng_to_cell(y, x, res) for x, y in zip(places.geometry.x, places.geometry.y)])
    hx["places"] = cells.value_counts().reindex(hx.index).fillna(0).astype(int)
    order = hx.sort_values(["places", "demand"], ascending=False, kind="stable").index
    hx["rank"] = pd.Series(np.arange(len(hx)), index=order)
    print(f"res {res}: {len(hx):,} area hexes, {int((hx.places > 0).sum()):,} with at least one place; "
          f"places in area hexes {hx.places.sum():,}; busiest hex {hx.places.max():,}")

def n_top(res, x_pct):
    """Number of candidate sites at X%: the top X% of the area's hexes, only those with a place."""
    hx = hexes[res]
    return min(math.ceil(x_pct / 100 * len(hx)), int((hx.places > 0).sum()))

cand_all = {res: hexes[res][hexes[res]["rank"] < n_top(res, max(X_GRID))].sort_values("rank") for res in RES}

### Build the travel-time table

For each resolution the server starts fresh (so the peak-memory reading is its own), the candidate sites are snapped
with `/nearest`, and OSRM's `/table` gives free-flow minutes for **slice candidates × all area hexes** (not only slice
hexes, so a candidate's pair count is not cut off at the slice edge; that is what the extrapolation uses). The
largest X in the grid is built once; smaller X are row subsets. `osrm-routed` allows at most 1,000 coordinates per
`/table` request (`--max-table-size 1000` in `osrm.sh serve`; the default is 100), so requests go in chunks of 50
sources × 950 destinations. Unroutable pairs come back null and are counted, then dropped. The table time below is the
wall time of all requests, JSON included; peak memory is the `osrm-routed` container's cgroup `memory.peak` after
the queries, which counts page cache too, so it is an upper bound.

In [ ]:
def get(url):
    with urllib.request.urlopen(url, timeout=300) as r:
        return json.load(r)

def snap(lon, lat):
    w = get(f"{OSRM}/nearest/v1/driving/{lon:.6f},{lat:.6f}?number=1")["waypoints"][0]
    return w["location"][0], w["location"][1], w["distance"]

def table(src_lon, src_lat, dst_lon, dst_lat, n_src=50, n_dst=950):
    """Free-flow minutes, sources x destinations (NaN where OSRM has no route), and how far each destination snapped."""
    out = np.full((len(src_lon), len(dst_lon)), np.nan, dtype=np.float32)
    dst_snap = np.zeros(len(dst_lon))
    for i in range(0, len(src_lon), n_src):
        s = slice(i, i + n_src)
        for j in range(0, len(dst_lon), n_dst):
            d = slice(j, j + n_dst)
            ns, nd = len(src_lon[s]), len(dst_lon[d])
            coords = ";".join(f"{x:.6f},{y:.6f}" for x, y in zip(np.r_[src_lon[s], dst_lon[d]], np.r_[src_lat[s], dst_lat[d]]))
            url = (f"{OSRM}/table/v1/driving/{coords}?sources={';'.join(map(str, range(ns)))}"
                   f"&destinations={';'.join(map(str, range(ns, ns + nd)))}&annotations=duration")
            res = get(url)
            assert res["code"] == "Ok", res
            out[s, d] = np.array(res["durations"], dtype=float) / 60   # null -> NaN
            if i == 0:
                dst_snap[d] = [w["distance"] for w in res["destinations"]]
    return out, dst_snap

def container_peak_mb():
    out = subprocess.run(["docker", "exec", "catchment-osrm", "cat", "/sys/fs/cgroup/memory.peak"], check=True, capture_output=True, text=True).stdout
    return int(out) / 1e6

D, cand, hex_snap, build = {}, {}, {}, {}   # per res: minutes matrix, snapped candidates, hex snap distances, timing and memory
try:
    for res in RES:
        _ = subprocess.run([str(NB / "osrm.sh"), "serve"], check=True)
        hx, c = hexes[res], cand_all[res].copy()
        idle_mb = container_peak_mb()
        c["snap_lon"], c["snap_lat"], c["snap_m"] = zip(*[snap(x, y) for x, y in zip(c.lon, c.lat)])
        sc = c[c.in_slice]
        t0 = time.perf_counter()
        D[res], hex_snap[res] = table(sc.snap_lon.to_numpy(), sc.snap_lat.to_numpy(), hx.lon.to_numpy(), hx.lat.to_numpy())
        build[res] = {"seconds": time.perf_counter() - t0, "n_src": len(sc), "idle_mb": idle_mb, "peak_mb": container_peak_mb()}
        cand[res] = c
finally:   # stop the server even if a query fails
    _ = subprocess.run([str(NB / "osrm.sh"), "stop"], check=True)

for res in RES:
    n_null = int(np.isnan(D[res]).sum())
    dead_hexes = int(np.isnan(D[res]).all(axis=0).sum())
    b = build[res]
    print(f"res {res}: {b['n_src']} slice candidates × {D[res].shape[1]:,} area hexes = {D[res].size:,} pairs in {b['seconds']:.1f} s "
          f"({b['seconds'] / b['n_src'] * 1000:.0f} ms per source); null (unroutable) pairs {n_null:,} "
          f"({n_null / D[res].size:.2%}), of which hexes with no route from any source: {dead_hexes}")

### Choosing X

Rule: **X is the smallest value in the grid whose slice candidates reach at least 99% of the slice's demand at the
default scenario** (promise 10, multiplier 1.4: reachable if free-flow min × 1.4 + 3 ≤ 10, so free-flow ≤ 5.0 min). If
none does, the largest value is used and a warning is printed. The rule is applied to each resolution on its own, and
the table lists every X so the choice can be read against the alternatives. "Candidates reach" counts only
candidates inside the slice, though a slice-edge hex might be reached from just outside it, so it is a slightly
strict test. Extrapolated pairs are the area's candidate count × the mean pairs per slice candidate, with every
pair counted to all area hexes.

In [ ]:
def slice_rows(res, x_pct):
    """Boolean mask over the rows of D[res] (slice candidates at the largest X) for the candidates at X%."""
    c = cand[res]
    return (c[c.in_slice]["rank"] < n_top(res, x_pct)).to_numpy()

rows = []
for res in RES:
    hx = hexes[res]
    in_sl, dem = hx.in_slice.to_numpy(), hx.demand.to_numpy()
    for x in X_GRID:
        m = slice_rows(res, x)
        Dm = D[res][m]
        reached = reachable(Dm[:, in_sl]).any(axis=0) if m.any() else np.zeros(in_sl.sum(), bool)
        pairs17 = (Dm <= MAX_PAIR_MIN).sum(axis=1).mean() if m.any() else np.nan
        pairs5 = reachable(Dm).sum(axis=1).mean() if m.any() else np.nan
        rows.append({"res": res, "X %": x, "candidates, area": n_top(res, x), "candidates, slice": int(m.sum()),
                     "slice demand reached": dem[in_sl][reached].sum() / dem[in_sl].sum(),
                     "pairs per candidate": pairs17, "extrapolated area pairs": n_top(res, x) * pairs17,
                     "extrapolated default pairs": n_top(res, x) * pairs5})
xtab = pd.DataFrame(rows)

X_CHOSEN = {}
for res in RES:
    ok = xtab[(xtab.res == res) & (xtab["slice demand reached"] >= REACH_TARGET)]
    if len(ok):
        X_CHOSEN[res] = float(ok["X %"].iloc[0])
    else:
        X_CHOSEN[res] = float(max(X_GRID))
        print(f"WARNING: res {res}: no X in the grid reaches {REACH_TARGET:.0%} of slice demand; using the largest, {X_CHOSEN[res]}%")
    print(f"res {res}: chosen X = {X_CHOSEN[res]}% -> {n_top(res, X_CHOSEN[res])} candidate sites in the area, {int(slice_rows(res, X_CHOSEN[res]).sum())} in the slice")

show_x = xtab.copy()
show_x["slice demand reached"] = (show_x["slice demand reached"] * 100).round(1).astype(str) + "%"
for col in ["pairs per candidate", "extrapolated area pairs", "extrapolated default pairs"]:
    show_x[col] = show_x[col].round(0)
display(show_x.style.hide(axis="index").format({"X %": "{:g}", "pairs per candidate": "{:,.0f}", "extrapolated area pairs": "{:,.0f}", "extrapolated default pairs": "{:,.0f}"}))

### What `02-plan` reads

`data/explore/candidates-r{9,8}.parquet`: the candidate sites in the area at the chosen X (hex id, hex centre, snapped
location, snap distance, places count, `in_slice`). `data/explore/table-r{9,8}.parquet`: the slice table in long
format, one row per slice candidate × slice hex pair within 17 min free-flow (`candidate`, `hex`, `minutes`). Both are
data, not committed.

In [ ]:
tables = {}
for res in RES:
    hx, c = hexes[res], cand[res]
    k = n_top(res, X_CHOSEN[res])
    chosen = c[c["rank"] < k]
    chosen[["lat", "lon", "snap_lat", "snap_lon", "snap_m", "places", "in_slice", "rank"]].to_parquet(EXPLORE / f"candidates-r{res}.parquet")
    m = slice_rows(res, X_CHOSEN[res])
    in_sl = hx.in_slice.to_numpy()
    r, col = np.nonzero(D[res][m][:, in_sl] <= MAX_PAIR_MIN)   # NaN compares False, so unroutable pairs drop out
    tables[res] = pd.DataFrame({
        "candidate": c[c.in_slice].index[m][r],
        "hex": hx.index[in_sl][col],
        "minutes": D[res][m][:, in_sl][r, col].astype(np.float32),
    })
    tables[res].to_parquet(EXPLORE / f"table-r{res}.parquet")
    print(f"res {res}: {len(chosen)} candidates, {len(tables[res]):,} slice-table pairs "
          f"({(EXPLORE / f'table-r{res}.parquet').stat().st_size / 1e6:.2f} MB on disk)")

### Snapping check

Carry-over from part 1: route 2 snapped within 49 m of a road but rode about 480 m out and back, probably from
landing on the wrong carriageway of a divided road. Pairs in the table can do the same, and the short ones are the
ones that decide who is reachable. Two checks. First, how far the candidate sites and the hex centres sit from the
nearest road. Second, on a random sample of 40 short pairs from each slice table (free-flow ≤ 6 min, fixed seed), a
`/route` call measures the out-and-back at each end the way part 1 does: the path ridden while still within 100 m
of the start, and after first coming within 100 m of the end, both measured from the snapped points here. About 100 m
or less is clean; more than 300 m is counted as a loop.

In [ ]:
def snap_stats(a):
    a = np.asarray(a, dtype=float)
    return {"n": len(a), "median m": np.median(a), "90th pct m": np.percentile(a, 90), "max m": a.max(),
            "> 50 m": int((a > 50).sum()), "> 100 m": int((a > 100).sum())}

snap_rows = {}
for res in RES:
    hx, c = hexes[res], cand[res]
    chosen = c[c["rank"] < n_top(res, X_CHOSEN[res])]
    snap_rows[f"res {res} candidate sites (area)"] = snap_stats(chosen.snap_m)
    snap_rows[f"res {res} hex centres (area)"] = snap_stats(hex_snap[res])
    snap_rows[f"res {res} hex centres (slice)"] = snap_stats(hex_snap[res][hx.in_slice.to_numpy()])
display(pd.DataFrame(snap_rows).T.round(0).astype({"n": int, "> 50 m": int, "> 100 m": int}))

def route_pair(src_lon, src_lat, dst_lon, dst_lat):
    url = f"{OSRM}/route/v1/driving/{src_lon},{src_lat};{dst_lon},{dst_lat}?overview=full&geometries=geojson"
    rt = get(url)
    assert rt["code"] == "Ok", rt
    (ax, ay), (bx, by) = (to_utm.transform(*w["location"]) for w in rt["waypoints"])
    x, y = to_utm.transform(*np.array(rt["routes"][0]["geometry"]["coordinates"]).T)
    along = np.r_[0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]
    leave = along[np.where(np.hypot(x - ax, y - ay) <= 100)[0].max()]
    arrive = along[-1] - along[np.where(np.hypot(x - bx, y - by) <= 100)[0].min()]
    return rt["routes"][0]["duration"] / 60, leave, arrive

samples = []
try:
    _ = subprocess.run([str(NB / "osrm.sh"), "serve"], check=True)
    for res in RES:
        hx, c = hexes[res], cand[res]
        short = tables[res][(tables[res].minutes <= 6) & (tables[res].candidate != tables[res].hex)]
        for _, p in short.sample(40, random_state=42).iterrows():
            s, h = c.loc[p.candidate], hx.loc[p.hex]
            minutes, leave, arrive = route_pair(s.snap_lon, s.snap_lat, h.lon, h.lat)
            samples.append({"res": res, "candidate": p.candidate, "hex": p.hex, "table min": p.minutes, "route min": minutes,
                            "leave m": leave, "arrive m": arrive, "hex snap m": hex_snap[res][hx.index.get_loc(p.hex)]})
finally:
    _ = subprocess.run([str(NB / "osrm.sh"), "stop"], check=True)

samples = pd.DataFrame(samples)
samples["start/end path m"] = samples[["leave m", "arrive m"]].max(axis=1)
loops = samples[samples["start/end path m"] > 300]
print(f"Sampled pairs (free-flow ≤ 6 min): {len(samples)}; max |route min - table min|: {(samples['route min'] - samples['table min']).abs().max():.2f}")
for res in RES:
    s, l = samples[samples.res == res], loops[loops.res == res]
    print(f"res {res}: median start/end path {s['start/end path m'].median():.0f} m; more than 300 m at an endpoint: {len(l)} of {len(s)} "
          f"(start end: {int((s['leave m'] > 300).sum())}, hex end: {int((s['arrive m'] > 300).sum())})")

### Figures

Candidate sites, then catchments. A catchment is the set of hexes a site reaches at the default scenario (promise
10 min, multiplier 1.4, so free-flow ≤ 5.0 min), drawn at res 9 from the table on the 4 sites nearest the named
places of part 1. The dashed circle has the same area as the catchment, to show the difference between catchments
that follow roads and a radius.

In [ ]:
def hex_poly(c):
    return Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)])

slice_gs = gpd.GeoSeries([slice_poly], crs=4326)
fig, axes = plt.subplots(1, 2, figsize=(12, 6.5), constrained_layout=True)
for ax, res in zip(axes, RES):
    hx = hexes[res][hexes[res].in_slice]
    g = gpd.GeoDataFrame(hx[["places"]], geometry=[hex_poly(c) for c in hx.index], crs=4326)
    g.plot(column="places", ax=ax, cmap="viridis", norm=PowerNorm(0.5, vmin=0, vmax=g.places.max()), linewidth=0.1, edgecolor="white")
    slice_gs.boundary.plot(ax=ax, color="black", linewidth=1)
    c = cand[res]
    cs = c[(c["rank"] < n_top(res, X_CHOSEN[res])) & c.in_slice]
    ax.plot(cs.snap_lon, cs.snap_lat, "o", color="red", markersize=4, markeredgecolor="white", markeredgewidth=0.5)
    ax.set_title(f"H3 res {res}: {len(cs)} candidate sites in the slice\n(top {X_CHOSEN[res]:g}% of area hexes, {n_top(res, X_CHOSEN[res])} in the area)")
    ax.set_aspect(1 / np.cos(np.radians(28.44)))
    ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
    sm = plt.cm.ScalarMappable(cmap="viridis", norm=PowerNorm(0.5, vmin=0, vmax=g.places.max()))
    fig.colorbar(sm, ax=ax, shrink=0.8, label="Overture places per hex")
fig.suptitle("Candidate sites (red, snapped to the road) over places per hex, slice")
fig.savefig(FIGS / "01-candidates.png", dpi=120)
plt.show()

![Candidate sites over places per hex, res 9 and res 8](figures/01-candidates.png)

In [ ]:
SITES = {"Old city (Sadar Bazar)": "Sadar Bazar", "Medanta": "Medanta", "Sohna Rd": "Sohna Rd, sec 48", "Golf Course Rd": "Golf Course Rd, sec 57"}
res = 9
hx, c = hexes[res], cand[res]
sc = c[c.in_slice]                                   # rows of D[9]
in_chosen = (sc["rank"] < n_top(res, X_CHOSEN[res])).to_numpy()
colors = plt.cm.Dark2(np.arange(len(SITES)))
cosl = np.cos(np.radians(28.44))

fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)
slice_gs.plot(ax=ax, facecolor="#f2f2f2", edgecolor="black", linewidth=1)
catch_rows = []
for (label, point), color in zip(SITES.items(), colors):
    px, py = POINTS[point]
    d = np.hypot((sc.snap_lon - px) * cosl, sc.snap_lat - py).to_numpy().copy()
    d[~in_chosen] = np.inf
    i = int(np.argmin(d))
    site = sc.iloc[i]
    reach = reachable(D[res][i])                    # NaN compares False
    cells = hx.index[reach]
    area_km2 = sum(h3.cell_area(h, "km^2") for h in cells)
    r_km = math.sqrt(area_km2 / math.pi)
    gpd.GeoSeries([hex_poly(h) for h in cells], crs=4326).plot(ax=ax, facecolor=color, edgecolor="none", alpha=0.5)
    t = np.linspace(0, 2 * np.pi, 200)
    ax.plot(site.snap_lon + r_km / (111.32 * cosl) * np.cos(t), site.snap_lat + r_km / 110.574 * np.sin(t), "--", color=color, linewidth=2)
    ax.plot(site.snap_lon, site.snap_lat, "*", color=color, markersize=16, markeredgecolor="black", zorder=5)
    ax.plot([], [], "s", color=color, alpha=0.6, label=f"{label}: {len(cells)} hexes, {area_km2:.1f} km² (dashed circle: radius {r_km:.1f} km)")
    catch_rows.append({"site": label, "site hex": sc.index[i], "km from named place": d[i] * 111.32, "hexes": len(cells), "km²": area_km2,
                       "demand reached (orders/day)": hx.demand.to_numpy()[reach].sum()})
ax.set_xlim(77.005 - 0.004, 77.105 + 0.004); ax.set_ylim(28.405 - 0.004, 28.480 + 0.004)
ax.set_aspect(1 / cosl)
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8, frameon=False)
ax.set_title(f"Catchments at promise {PROMISE} min, multiplier {MULTIPLIER} (res 9; stars are candidate sites)")
fig.savefig(FIGS / "01-catchments.png", dpi=120, bbox_inches="tight")
plt.show()
display(pd.DataFrame(catch_rows).round({"km from named place": 2, "km²": 1, "demand reached (orders/day)": 0}))

![Catchments at the default scenario against circles of the same area](figures/01-catchments.png)

### Pair count and extrapolation to the full area

The slice table's pair count, then the full area: **area candidates × the mean pairs per slice candidate**, each counted
to all area hexes. Two counts: all pairs within 17 min free-flow (the stored table), and the default-scenario pairs
within 5.0 min free-flow, which the solver assigns from. Table time scales per source (the area has more sources, each
to all area hexes; the graph is larger there too, so this is a floor). The in-memory size is the full-area table as
int32 candidate, int32 hex and float32 minutes in long format, 12 bytes a pair. Caveat: the slice is denser than the
area's average (the area includes sparse outskirts), so mean pairs per candidate here probably over-estimate the
area's, most at its edge.

In [ ]:
ext = {}
for res in RES:
    hx, b = hexes[res], build[res]
    k, m = n_top(res, X_CHOSEN[res]), slice_rows(res, X_CHOSEN[res])
    Dm = D[res][m]
    p17, p5 = (Dm <= MAX_PAIR_MIN).sum(axis=1).mean(), reachable(Dm).sum(axis=1).mean()
    full17, full5 = k * p17, k * p5
    ext[f"res {res}"] = {
        "hexes, area / slice": f"{len(hx):,} / {int(hx.in_slice.sum()):,}",
        "X (% of area hexes)": f"{X_CHOSEN[res]:g}",
        "candidates, area / slice": f"{k:,} / {int(m.sum()):,}",
        "slice table pairs (≤ 17 min)": f"{len(tables[res]):,}",
        "mean pairs per slice candidate, to all area hexes (≤ 17 / ≤ 5 min)": f"{p17:,.0f} / {p5:,.0f}",
        "full-area pairs (≤ 17 min), extrapolated": f"{full17:,.0f}",
        "full-area default-scenario pairs (≤ 5 min), extrapolated": f"{full5:,.0f}",
        "full-area table in memory (≤ 17 min, 12 B a pair)": f"{full17 * 12 / 1e6:,.1f} MB",
        "full-area default-scenario pairs in memory": f"{full5 * 12 / 1e6:,.1f} MB",
        "OSRM table time per source, to all area hexes": f"{b['seconds'] / b['n_src'] * 1000:,.0f} ms",
        "OSRM table time, full area, extrapolated": f"{k * b['seconds'] / b['n_src']:,.0f} s",
        "osrm-routed peak memory after the queries (upper bound, MB)": f"{b['peak_mb']:,.0f} (idle {b['idle_mb']:,.0f})",
    }
display(pd.DataFrame(ext))

Results from the run on 2026-10-29, part 2 (copied here because outputs are stripped on commit; re-running updates
the printed tables, not this one):

**Candidate sites.** 81,939 Overture places (3 permanently_closed dropped); 3,159 of the 4,859 res-9 area hexes and
616 of the 697 res-8 ones hold at least one. Slice demand reached at the default scenario (free-flow ≤ 5.0 min), by X:

| X (% of area hexes) | 2.5 | 5 | 10 | 15 | 20 | 30 |
|---|---:|---:|---:|---:|---:|---:|
| res 9: candidates, area / slice | 122 / 53 | 243 / 108 | 486 / 208 | 729 / 308 | 972 / 418 | 1,458 / 554 |
| res 9: slice demand reached | 79.9% | 91.7% | 96.0% | 97.0% | 97.3% | **99.3%** |
| res 8: candidates, area / slice | 18 / 8 | 35 / 15 | 70 / 36 | 105 / 56 | 140 / 70 | 210 / 84 |
| res 8: slice demand reached | 34.8% | 45.2% | 72.3% | 86.7% | 92.5% | 96.0% |

**Chosen X = 30% at res 9**: 1,458 candidate sites in the area, 554 in the slice (it is the only grid value that
reaches 99%, at 99.3%). **At res 8 no X reaches 99%**, so the rule falls back to the largest, 30%: 210 candidate sites
in the area, 84 in the slice, 96.0% reached. The data does not argue for a different X at res 8, so both use 30%. Two
things to know. The last 3 points of coverage at res 9 (96.0% at 10% to 99.3% at 30%) cost three times the
candidates, so 99% is a stiff target; at X = 10% the full-area table would be about 0.5 M pairs, not 1.4 M. And at
30% the rule is close to "every hex with some places": 554 of the slice's 713 res-9 hexes are candidates, because the
slice holds 38% of the area's candidates in 15% of its hexes. The res-8 gap is a coarseness effect: a res-8 hex is
about 0.74 km², so its centre is often more than 5 free-flow minutes from the next candidate's.

**Table, snapping and catchments** (res 9; res 8 in the table below). Nothing was unroutable: no null pair and no hex
without a route (0 of 2,691,886 and of 58,548 slice-candidate × area-hex pairs). Candidate sites snap a median 13 m
(res 9) and 14 m (res 8) from the road, 90th percentile about 50 m, worst 218 m; 23 of 1,458 are over 100 m away.
Hex centres in the slice snap a median 13 m (23 of 713 over 100 m, worst 294 m); across the whole area the median is
33 m, 1,231 of 4,859 are over 100 m and the worst is 1,075 m, in open land at the area's edge. On 40 random short pairs
(free-flow ≤ 6 min) per resolution, the table's minutes match `/route` exactly. **6 of 40 (res 9) and 5 of 40 (res 8)
ride more than 300 m out and back at an endpoint**: res 9 has 2 at the candidate end and 4 at the hex end, res 8 has 3
and 2; the median is 114-116 m, so most are clean. That is about one short pair in seven. It is **an open point for stage 1**: nothing is fixed here, and the options in #4 (`approaches=curb`, or snapping
candidate sites to a non-trunk road) are untried. The four catchments at the default scenario (figure above) cover 40
to 57 hexes, 4.4 to 6.2 km², and 889 to 1,411 orders a day of modelled demand: far smaller than the 17 km² the slice choice
(decision log) assumed for a 10-minute catchment, because the promise leaves 5 free-flow minutes after the multiplier
and allowance. A circle of the same area is a fair first guess in dense blocks, but the catchments run along roads
and stop at barriers (the tendrils and gaps in the figure).

| | res 9 | res 8 |
|---|---:|---:|
| hexes, area / slice | 4,859 / 713 | 697 / 102 |
| X (% of area hexes) | 30 | 30 |
| candidates, area / slice | 1,458 / 554 | 210 / 84 |
| slice table pairs (≤ 17 min) | 299,666 | 6,454 |
| mean pairs per slice candidate, to all area hexes (≤ 17 / ≤ 5 min) | 959 / 37 | 135 / 5 |
| full-area pairs (≤ 17 min), extrapolated | 1,398,401 | 28,408 |
| full-area default-scenario pairs (≤ 5 min), extrapolated | 54,246 | 1,105 |
| full-area table in memory (≤ 17 min, 12 B a pair) | 16.8 MB | 0.3 MB |
| OSRM table time per source, to all area hexes | 30 ms | 4 ms |
| OSRM table time, full area, extrapolated | 44 s | 1 s |
| `osrm-routed` peak memory after the queries (upper bound, idle) | 409 MB (161) | 198 MB (160) |

**Reading the table.** Res 9 fits easily: the full-area table is about 1.4 M pairs and 17 MB in memory, and the
solver's default-scenario assignment pairs about 54 k. The table time varies with machine load (an earlier
run of the same cell took 99 ms a source at res 9, so about 144 s for the area); the peak memory barely moved
(407 MB). The server's own idle footprint is about 160 MB, so the queries add about 250 MB at most. Res 8 is far cheaper but misses
the coverage target; the resolution choice (finest that fits)
therefore points at **res 9** on table size. Whether CP-SAT solves 54 k assignment pairs in 10 s is `02-plan`'s
question. Caveats: the slice is denser than the area's average, so mean pairs per candidate here probably
over-estimate the area's, most at its edge (the area is not truncated at the slice edge, but its sparse outskirts
have fewer hexes within 17 min); the full-area graph is larger than the slice's, so table time is a floor; the
slice-edge 99% test ignores candidates just outside the slice.

## Next

`02-plan` (#5) reads `data/explore/hexes-r{9,8}.parquet` (from `00-demand`), `candidates-r{9,8}.parquet` and
`table-r{9,8}.parquet` (from this notebook), places a placeholder synthetic network, and runs greedy and CP-SAT on the
slice. Snapping onto the wrong carriageway is an open point for stage 1.